# Notebook 03 — Why a registry

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS — Workshop 2

---

If MCP servers are just Lambdas, why do I need an Agent Registry?

This notebook answers that question by registering the five MCP servers and five
Phase 1 agents, then querying the registry as two different personas to see
persona-scoped discovery in action.

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Agent Registry** | A central catalog of all agents and MCP servers in the system. Answers the question "what capabilities exist and who can use them?" Used by UIs for capability palette population and by governance teams for approval workflows. |
| **Capability palette** | The set of actions a UI renders for the current user. Populated live from the registry, not hardcoded. Different personas see different palettes because the registry filters by persona claim. |
| **Persona-scoped discovery** | The registry returns only capabilities that the requesting persona is allowed to discover. A Consumer Banker sees referral-related agents; a Wealth Advisor sees portfolio-related agents. The filtering happens at the registry, not at the UI. |
| **Registration payload** | The JSON descriptor submitted to the registry when an agent or MCP server is registered. Contains the name, version, input/output schemas, and the `discoverable_by` list that controls persona-scoped discovery. |
| **Governance gate** | The approval workflow that a new agent must pass before it appears in the registry. Every registered agent is a new MRM submission, a new attack surface, and a new auditable component. The registry is the gate. |

## Discovery and governance in one place

Notebook 02 built five MCP servers — stable typed interfaces that agents call
instead of calling AWS services directly. Those servers exist as Lambda functions
now, but nothing in the system *knows* they exist. If you wanted to build a UI
that shows a banker what actions are available, you would have to hardcode the
list. If a new MCP server were deployed tomorrow, the UI would not know about it
until a developer added it to the code and redeployed. That is the first problem
a registry solves: discovery.

The second problem is governance. In a regulated bank, every new agent is a new
model risk submission (SR 11-7 / OCC 2011-12). Every new MCP server is a new
attack surface. An organization that lets developers wire agents directly into
applications has no central place to review what agents exist, who can invoke
them, or which ones touch regulated data. The registry is that central place.
Registration is the approval gate: if it is not in the registry, it does not
exist to the system.

AWS Agent Registry (part of Bedrock AgentCore) provides both capabilities. When
you register an agent, you submit a descriptor that declares its name, version,
input/output schemas, and — critically — a `discoverable_by` list that names
which persona claims are allowed to see it. When a UI queries the registry with
a persona claim, the registry returns only the agents and MCP servers whose
`discoverable_by` list includes that claim. This is persona-scoped discovery,
and it is the mechanism behind the capability palette you will see in notebook 05.

This is Thesis 1 from the ATLAS architecture: registry-first agent discovery.
The UI never hardcodes what agents exist. It asks the registry. The registry
answers based on who is asking. The result is a system where adding a new agent
requires only a registration (with governance approval) — not a UI redeploy.

In this notebook, you will register all five MCP servers and all five Phase 1
agents. Then you will query the registry as a Consumer Banker and as a Wealth
Advisor, and observe that the two personas see different capability sets. The
substrate is identical; the registry filters produce different views.

In [ ]:
import sys
import os
import json

# Workshop 1's shared helpers
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

# Workshop 2's spec directory — we read the JSON descriptors from here
SPEC_DIR = "../../spec/04-aws-agent-registry"

print("Setup complete. Spec directory:", os.path.abspath(SPEC_DIR))

In [ ]:
# Build cell 1 — Load all JSON descriptors from the spec directory.
#
# Each agent and MCP server has a paired .json file that declares its
# registration payload. We load them all so we can register them in sequence.

def load_descriptors(subdir):
    """Load all .json descriptors from a spec subdirectory."""
    path = os.path.join(SPEC_DIR, subdir)
    descriptors = []
    for f in sorted(os.listdir(path)):
        if f.endswith(".json"):
            with open(os.path.join(path, f)) as fh:
                descriptors.append(json.load(fh))
    return descriptors

mcp_descriptors = load_descriptors("mcp-servers")
agent_descriptors = load_descriptors("agents")

print(f"Loaded {len(mcp_descriptors)} MCP server descriptors")
print(f"Loaded {len(agent_descriptors)} agent descriptors")
print()
print("MCP servers:", [d['mcp_server_name'] for d in mcp_descriptors])
print("Agents:", [d['agent_name'] for d in agent_descriptors])

In [ ]:
# Build cell 2 — Register MCP servers with the Agent Registry.
#
# In a deployed environment, this calls boto3 bedrock-agentcore.
# For this notebook, we simulate registration with a local registry
# dict so you can see the mechanics without needing a live service.

local_registry = {"mcp_servers": {}, "agents": {}}

for desc in mcp_descriptors:
    name = desc["mcp_server_name"]
    local_registry["mcp_servers"][name] = {
        "name": name,
        "version": desc["version"],
        "operations": list(desc["operations"].keys()),
        "discoverable_by": desc.get("registry_metadata", {}).get("discoverable_by", []),
    }
    print(f"Registered MCP server: {name} (v{desc['version']})")

print(f"\n{len(local_registry['mcp_servers'])} MCP servers registered.")

In [ ]:
# Build cell 3 — Register Phase 1 agents with the Agent Registry.
#
# Same pattern as MCP servers. Each agent's discoverable_by list
# determines which personas can see it.

phase_1_agents = [d for d in agent_descriptors if d.get("phase") == 1]

for desc in phase_1_agents:
    name = desc["agent_name"]
    local_registry["agents"][name] = {
        "name": name,
        "version": desc["version"],
        "posture": desc["posture"],
        "display_name": desc.get("registry_metadata", {}).get("display_name", name),
        "discoverable_by": desc.get("registry_metadata", {}).get("discoverable_by", []),
    }
    print(f"Registered agent: {name} (posture: {desc['posture']})")

print(f"\n{len(local_registry['agents'])} agents registered.")

In [ ]:
# Build cell 4 — Persona-scoped discovery function.
#
# This is what the UI calls: "given my persona, what can I do?"
# The registry filters its response by the persona claim.

def discover_capabilities(persona_claim):
    """Return capabilities discoverable by the given persona."""
    agents = [
        a for a in local_registry["agents"].values()
        if persona_claim in a["discoverable_by"]
    ]
    mcp_servers = [
        m for m in local_registry["mcp_servers"].values()
        if persona_claim in m["discoverable_by"]
    ]
    return {"agents": agents, "mcp_servers": mcp_servers}

print("Discovery function ready.")

## Verification

Two things must be true for the registry to be working correctly:

1. A Consumer Banker discovers the five Phase 1 agents (all are discoverable by `atlas-consumer-banker`).
2. A Wealth Advisor discovers a *different* set — specifically, agents whose `discoverable_by` includes `atlas-wealth-advisor` but not those restricted to Consumer Bankers only.

The difference proves that discovery is persona-scoped. The UI does not need to
know which agents exist — it asks the registry, and the registry answers based on
who is asking.

In [ ]:
# Verification cell 1 — Consumer Banker discovery.
#
# Expected: all 5 Phase 1 agents are discoverable.
# If this fails: check that the agent descriptors have
# 'atlas-consumer-banker' in their discoverable_by lists.

consumer_caps = discover_capabilities("atlas-consumer-banker")

print("Consumer Banker capability palette:")
print(f"  Agents ({len(consumer_caps['agents'])}):\n")
for a in consumer_caps["agents"]:
    print(f"    • {a['display_name']} ({a['name']}, posture: {a['posture']})")

print(f"\n  MCP servers ({len(consumer_caps['mcp_servers'])}):\n")
for m in consumer_caps["mcp_servers"]:
    print(f"    • {m['name']} — operations: {m['operations']}")

# Assertion
assert len(consumer_caps["agents"]) >= 4, \
    f"Expected at least 4 agents for Consumer Banker, got {len(consumer_caps['agents'])}"
print("\n✓ Consumer Banker sees the expected capability palette.")

In [ ]:
# Verification cell 2 — Wealth Advisor discovery.
#
# Expected: a DIFFERENT set than the Consumer Banker.
# Specifically, referral-rationale-drafter is Consumer Banker only.
# If this fails: check the discoverable_by lists in the JSON descriptors.

advisor_caps = discover_capabilities("atlas-wealth-advisor")

print("Wealth Advisor capability palette:")
print(f"  Agents ({len(advisor_caps['agents'])}):\n")
for a in advisor_caps["agents"]:
    print(f"    • {a['display_name']} ({a['name']}, posture: {a['posture']})")

# The key assertion: the two palettes are different
consumer_agent_names = {a["name"] for a in consumer_caps["agents"]}
advisor_agent_names = {a["name"] for a in advisor_caps["agents"]}

print(f"\n  Consumer Banker sees: {sorted(consumer_agent_names)}")
print(f"  Wealth Advisor sees:  {sorted(advisor_agent_names)}")
print(f"  Difference:           {sorted(consumer_agent_names - advisor_agent_names)}")

assert consumer_agent_names != advisor_agent_names, \
    "Palettes should differ — discovery is persona-scoped"

# referral-rationale-drafter should be Consumer Banker only
assert "referral-rationale-drafter" not in advisor_agent_names, \
    "Wealth Advisor should NOT see referral-rationale-drafter"

print("\n✓ Persona-scoped discovery confirmed: different personas see different palettes.")

## What just changed

You have registered all five MCP servers and all five Phase 1 agents in the Agent
Registry. You have confirmed that discovery is persona-scoped: a Consumer Banker
and a Wealth Advisor see different capability palettes from the same registry.

This is the foundation of Thesis 1 (registry-first agent discovery). The UI you
will build in notebook 05 does not hardcode which agents exist — it queries the
registry and renders whatever comes back. Adding a new agent to the system
requires only a registration, not a UI redeploy.

The next notebook connects the registry to a FIBO-shaped GraphQL API — the schema
that the UI will write components against.